# Open Vocabulary

Open-vocabulary segmentation labels a scene with **any word**. The model returns one unit-length embedding per point,
$(N, \text{embed\_dim})$, in the space of a CLIP text encoder, and a point takes the prompt whose text embedding is
nearest by cosine similarity. There is no class list: the vocabulary is written at inference time.

![Room input and the points answering three prompts](../assets/tasks/open_vocabulary.png)

## Run a pretrained checkpoint

The example reads one room of the [`ScanNet20`](../api/datasets/scannet.md) validation set. ScanNet requires a signed agreement: request it on its [download page](https://kaldir.vc.in.tum.de/scannet/) and extract it under `data/ScanNet/raw/`.

The [OpenScene](../api/models/openscene.md) checkpoints read geometry only, so `x` is `None` and the registered transform voxelizes the room at 2 cm. `info["weights"]["text_model"]` names the CLIP checkpoint the embeddings are aligned with, and `info["weights"]["text_revision"]` its commit. The sparse convolutions need a GPU, and the model runs in float32.

In [ ]:
import torch

import torch_pointcloud as tp
from torch_pointcloud.datasets import ScanNet20
from torch_pointcloud.utils.data import collate

model, info = tp.create_model(
    "openscene-openseg.scannet20.songyou-peng",
    task="open-vocabulary-segmentation",
    pretrained=True,
    return_info=True,
)
model = model.cuda().eval()

# Load one room: the transform voxelizes it and keeps the source cloud under `origin_*`
dataset = ScanNet20(root="data", split="val", transform=info["transform"], use_axis_alignment=False)
sample = dataset[0]

data = collate([sample])
with torch.no_grad():
    embeddings = model(None, data["pos_grid"].cuda(), data["batch"].cuda())

print(f"Embeddings: {tuple(embeddings.shape)}, aligned with {info['weights']['text_model']}")
# Embeddings: (164832, 768), aligned with openai/clip-vit-large-patch14-336

## Query the scene with text

[`CLIPTextEncoder`](../api/layers/clip_text_encoder.md) embeds prompts with the same CLIP checkpoint at the same commit (the optional `transformers` package reads it from the Hub). The checkpoint was scored with the prompt template `info["weights"]["prompt_template"]`, `"a {} in a scene"`. Point and text embeddings are unit vectors, so their product is the cosine similarity, a per-point score for that prompt.

In [ ]:
from torch_pointcloud.datasets.scannet import SCANNET20_CLASSES
from torch_pointcloud.layers import CLIPTextEncoder

text_encoder = CLIPTextEncoder(info["weights"]["text_model"], revision=info["weights"]["text_revision"]).cuda()
template = info["weights"]["prompt_template"]
queries = ["chair", "window", "a place to sit"]
with torch.no_grad():
    text = text_encoder([template.format(query) for query in queries])  # (3, 768), unit norm

similarity = embeddings @ text.t()  # (N, 3) in [-1, 1]

# the checkpoint voxelizes, so check against the labels the model actually saw
segment = data["segment"].cuda()
for j, query in enumerate(queries):
    top = similarity[:, j].topk(1000).indices
    labels, counts = torch.unique(segment[top], return_counts=True)
    print(f"{query!r}: {counts.max().item() / 10:.1f}% of the 1000 best points are {SCANNET20_CLASSES[labels[counts.argmax()]]!r}")
# 'chair': 100.0% of the 1000 best points are 'chair'
# 'window': 80.3% of the 1000 best points are 'window'
# 'a place to sit': 99.0% of the 1000 best points are 'chair'

`"a place to sit"` is not a ScanNet class and the model never saw a label: the chairs answer it because their points sit next to the text in CLIP's space.

## Zero-shot segmentation

A vocabulary of prompts turns the similarities into logits: the `argmax` over the 20 ScanNet classes is a zero-shot segmentation, scored like any other. Predictions are per voxel, so `inverse` scatters them back to every point of the source cloud. `examples/openscene/open_vocabulary_segmentation_scannet_benchmark.py` scores the validation set the same way, with the authors' wording for two prompts (`"shower curtain"` and a bare `"other"`). On one room, every class the room does not contain scores an IoU of 0 in the mean.

In [ ]:
from torch_pointcloud.metrics import accuracy, confusion_matrix, intersection_over_union

prompts = [template.format(name) for name in SCANNET20_CLASSES]
with torch.no_grad():
    vocabulary = text_encoder(prompts)

logits = embeddings @ vocabulary.t()  # (N, 20)
preds = logits.argmax(dim=1)[data["inverse"].cuda()].cpu()

cm = confusion_matrix(preds, data["origin_segment"], num_classes=len(prompts), ignore_index=-1)
print(f"accuracy {accuracy(cm):.4f}")
print(f"mIoU     {intersection_over_union(cm):.4f}")
# accuracy 0.5562
# mIoU     0.1793

The checkpoint never saw a ScanNet label: the network was distilled from per-pixel OpenSeg features of the posed RGB frames. Its validation scores, one pass and the authors' five randomly rotated and scaled passes, are listed under [Checkpoints](#checkpoints).

## Inputs and outputs

| Argument    | Shape                              | Description                                                                 |
| ----------- | ---------------------------------- | --------------------------------------------------------------------------- |
| `x`         | $(N, C)$ or `None`                 | Per-point features; the released checkpoints read geometry only (`None`)   |
| `pos_grid`  | $(N, 3)$                           | Integer voxel-grid coordinates                                              |
| `batch`     | $(N,)$                             | Index tensor associating each point with its point cloud                    |
| **returns** | $(N, \text{embed\_dim})$           | One unit-length embedding per point in the text space                     |

`model.embed_dim` is the width of that space; `info["weights"]["text_model"]`, `info["weights"]["text_revision"]` and `info["weights"]["prompt_template"]` are the CLIP checkpoint, its commit and the prompt to encode a vocabulary with. `forward_head(..., pre_logits=True)` returns the $(N, \text{num\_features})$ decoder features entering the projection.

## Outdoor LiDAR

`openscene-openseg.nuscenes.songyou-peng` was distilled on 0.5 s of LiDAR sweeps, `NuScenes(max_sweeps=10)`, and voxelizes at 5 cm. The same steps run on a [`NuScenes`](../api/datasets/nuscenes.md) keyframe with outdoor prompts (`"a car in a scene"`, `"a road in a scene"`, ...).